## Spatial transcriptomics preprocessing

In these GettingStarted notebooks, we will guide you through the process of preprocessing, training, and performing inference with DeepSpot on your spatial transcriptomics data. First, we will begin with data preprocessing. In the training notebook, we will demonstrate how to train DeepSpot, adjust hyperparameters, and export the model weights. Finally, in the inference notebook, we will show you how to load the model weights and perform spatial transcriptomics prediction using H&E images.

In [2]:
import os
os.chdir('../../')

Export packages

In [3]:
from deepspot.utils.utils_image import get_morphology_model_and_preprocess
from deepspot.utils.utils_image import compute_mini_tiles
from deepspot.utils.utils_image import crop_tile

from pathlib import Path
from tqdm import tqdm
import scanpy as sc
import pandas as pd
import numpy as np
import pyvips
import torch
import glob
import yaml
import json

/scratch2/ig76/sasunih/conda/envs/deepspot/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Specify the input files. For this example, we have selected one sample from the COAD dataset [1], which was downloaded using the HEST1K pipeline [2]. You can modify this pipeline to process multiple samples. The goal is to help you understand the underlying logic.

[1] Valdeolivas, A., Amberg, B., Giroud, N., Richardson, M., Gálvez, E. J., Badillo, S., ... & Hahn, K. (2023). Charting the heterogeneity of colorectal cancer consensus molecular subtypes using spatial transcriptomics. bioRxiv, 2023-01.

[2] Jaume, G., Doucet, P., Song, A. H., Lu, M. Y., Almagro-Pérez, C., Wagner, S. J., ... & Mahmood, F. (2024). Hest-1k: A dataset for spatial transcriptomics and histology image analysis. arXiv preprint arXiv:2406.16192.

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [5]:
n_mini_tiles = 9 # number of non-overlaping sub-spots per subspot
image_feature_model = "inception" # foundation model 
sample = "ZEN38" # COAD dataset sample
out_folder = "example_data"
adata_in = f"example_data/data/h5ad/{sample}.h5ad"
json_path = f"example_data/data/meta/{sample}.json"
img_path = f"example_data/data/image/{sample}.jpg"

In [6]:
# create folder to save tile embeddings
folder_to_create = f"{out_folder}/data/image_features/{image_feature_model}/{sample}"
Path(folder_to_create).mkdir(parents=True, exist_ok=True)

Path(f"{out_folder}/data/inputX").mkdir(parents=True, exist_ok=True)

In [7]:
spot_diameter_fullres = round(json.load(open(json_path))["spot_diameter_fullres"]) # spot diameter
spot_diameter_fullres

60

We start by loading the spatial transcriptomics data and selecting the most variable genes

In [8]:
adata = sc.read_h5ad(adata_in)
adata

AnnData object with n_obs × n_vars = 387 × 19788
    obs: 'array_row', 'array_col', 'pxl_col_in_fullres', 'pxl_row_in_fullres', 'in_tissue', 'pxl_row_in_fullres_old', 'pxl_col_in_fullres_old', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'leiden', 'x_pixel', 'y_pixel', 'x_array', 'y_array', 'barcode', 'ground_truth'
    var: 'gene_ids', 'feature_types', 'genome', 'mito', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'gene_symbol', 'gene_name'
    uns: 'spatial'
    obsm: 'spatial'

In [9]:
sc.pp.highly_variable_genes(adata, flavor='seurat_v3_paper', 
                            n_top_genes=5000)
adata

AnnData object with n_obs × n_vars = 387 × 19788
    obs: 'array_row', 'array_col', 'pxl_col_in_fullres', 'pxl_row_in_fullres', 'in_tissue', 'pxl_row_in_fullres_old', 'pxl_col_in_fullres_old', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'leiden', 'x_pixel', 'y_pixel', 'x_array', 'y_array', 'barcode', 'ground_truth'
    var: 'gene_ids', 'feature_types', 'genome', 'mito', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'gene_symbol', 'gene_name', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'spatial', 'hvg'
    obsm: 'spatial'

We select the most highly variable genes in the isPredicted variable and save this table. Later, it is useful to understand which genes are the most variable (e.g., based on their rank).

In [10]:
adata.var["isPredicted"] = adata.var.highly_variable.values
adata.var["gene_name"] = adata.var.index.values
adata.var

,gene_ids,feature_types,genome,mito,n_cells_by_counts,mean_counts,log1p_mean_counts,pct_dropout_by_counts,total_counts,log1p_total_counts,n_counts,gene_symbol,gene_name,highly_variable,highly_variable_rank,means,variances,variances_norm,isPredicted
AL627309.1,ENSG00000238009,Gene Expression,GRCh38,False,2,0.005168,0.005155,99.483204,2.0,1.098612,2.0,AL627309.1,AL627309.1,False,NaN,0.005168,0.005155,0.980687,False
AL627309.5,ENSG00000241860,Gene Expression,GRCh38,False,3,0.007752,0.007722,99.224806,3.0,1.386294,3.0,AL627309.5,AL627309.5,False,NaN,0.007752,0.007712,0.974782,False
AP006222.2,ENSG00000286448,Gene Expression,GRCh38,False,2,0.005168,0.005155,99.483204,2.0,1.098612,2.0,AP006222.2,AP006222.2,False,NaN,0.005168,0.005155,0.980687,False
LINC01409,ENSG00000237491,Gene Expression,GRCh38,False,41,0.108527,0.103032,89.405685,42.0,3.761200,42.0,LINC01409,LINC01409,False,NaN,0.108527,0.102181,0.864774,False
FAM87B,ENSG00000177757,Gene Expression,GRCh38,False,1,0.002584,0.002581,99.741602,1.0,0.693147,1.0,FAM87B,FAM87B,False,NaN,0.002584,0.002584,0.999763,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
AC011043.1,ENSG00000276256,Gene Expression,GRCh38,False,4,0.010336,0.010283,98.966408,4.0,1.609438,4.0,AC011043.1,AC011043.1,False,NaN,0.010336,0.010256,0.971224,False
AL354822.1,ENSG00000278384,Gene Expression,GRCh38,False,2,0.005168,0.005155,99.483204,2.0,1.098612,2.0,AL354822.1,AL354822.1,False,NaN,0.005168,0.005155,0.980687,False
AL592183.1,ENSG00000273748,Gene Expression,GRCh38,False,4,0.010336,0.010283,98.966408,4.0,1.609438,4.0,AL592183.1,AL592183.1,False,NaN,0.010336,0.010256,0.971224,False
AC136616.1,ENSG00000273554,Gene Expression,GRCh38,False,52,0.170543,0.157467,86.563307,66.0,4.204693,66.0,AC136616.1,AC136616.1,True,1058.0,0.170543,0.235089,1.239278,True


In [11]:
adata.var.to_csv(f"{out_folder}/data/info_highly_variable_genes_Visium.csv", index=False)

Here, we save only the counts and store them as a pickle file to make them quickly accessible during training.

In [12]:
counts = pd.DataFrame(adata.X, index=adata.obs_names.values, columns=adata.var.index)
counts.index = [f"{b}_{sample}" for b in counts.index]
counts

,AL627309.1,AL627309.5,AP006222.2,LINC01409,FAM87B,LINC01128,LINC00115,FAM41C,LINC02593,SAMD11,...,MT-ND6,MT-CYB,BX004987.1,AC145212.1,MAFIP,AC011043.1,AL354822.1,AL592183.1,AC136616.1,AC007325.4
AAACCGTTCGTCCAGG-1_ZEN38,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,93.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAAGGCTCTCGCGCCG-1_ZEN38,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,45.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAAGGGATGTAGCAAG-1_ZEN38,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,11.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAATTAACGGGTAGCT-1_ZEN38,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,19.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
AACCGAGCTTGGTCAT-1_ZEN38,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,56.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTGGGACACTGCCCGC-1_ZEN38,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,18.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TTGGTCACACTCGTAA-1_ZEN38,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,19.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TTGTAAGGCCAGTTGG-1_ZEN38,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,22.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TTGTAATCCGTACTCG-1_ZEN38,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,19.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
counts.to_pickle(f"{out_folder}/data/inputX/{sample}.pkl")

We load the pathology foundation model along with its preprocessing pipeline and feature dimensions. Currently, we support Phikon, Uni, DenseNet121, ResNet50, and Inception. However, this list can be extended to include more models by modifying the `get_morphology_model_and_preprocess` function.

In [14]:
morphology_model, preprocess, feature_dim = get_morphology_model_and_preprocess(model_name=image_feature_model, 
                                                                                device=device)
feature_dim

Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /home/sasunih/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 238MB/s] 


2048

We now begin extracting tile representations and store them to save time during training by using the precomputed data. The representations are stored per spot, which includes the main representation and the k non-overlapping subspots, resulting in a shape of `(n_spots, k+1, feature_dim)`.

In [15]:
image = pyvips.Image.new_from_file(img_path)
morphology_model = morphology_model.to(device)
barcode = adata.obs_names
x_pixel = adata.obs.x_pixel
y_pixel = adata.obs.y_pixel


image = pyvips.Image.new_from_file(img_path)
main_features = np.zeros([len(adata), feature_dim])

In [16]:
for i, (b, x, y) in tqdm(enumerate(zip(barcode, x_pixel, y_pixel))): 

    main_tile = crop_tile(image, x, y, spot_diameter_fullres)
    preprocess_main_tile = preprocess(main_tile)

    X = np.zeros([n_mini_tiles + 1, 3, preprocess_main_tile.shape[1], preprocess_main_tile.shape[1]]) 
    X[0, :] = preprocess_main_tile

    mini_tiles = compute_mini_tiles(main_tile, n_mini_tiles)
    
    for j, mini_tile in enumerate(mini_tiles):
        
        X[j+1, :] = preprocess(mini_tile)

    
    X = torch.from_numpy(X)
    X = X.to(device).float()
    # We recommend using mixed precision for faster inference.
    with torch.autocast(device_type="cuda", dtype=torch.float32):
        with torch.inference_mode():
            output = morphology_model(X)
            output = output.float().detach().cpu().numpy()

    main_features[i,:] = output[0]
    
    np.save(f"{folder_to_create}/{b}.npy", output)

387it [00:26, 14.55it/s]


In [17]:
glob.glob(f"{out_folder}/data/image_features/{image_feature_model}/{sample}/*")[:10]

['example_data/data/image_features/inception/ZEN38/TCGTAAGACGACATTG-1.npy',
 'example_data/data/image_features/inception/ZEN38/CGGGCCTTCTTTGTAA-1.npy',
 'example_data/data/image_features/inception/ZEN38/AACGCGGTCTCCAGCC-1.npy',
 'example_data/data/image_features/inception/ZEN38/GCTAGCTTGAATAGCT-1.npy',
 'example_data/data/image_features/inception/ZEN38/TATTTATACCGAGTAG-1.npy',
 'example_data/data/image_features/inception/ZEN38/ATAAGTTACCGCGACG-1.npy',
 'example_data/data/image_features/inception/ZEN38/GTGAGTGGTACAACGC-1.npy',
 'example_data/data/image_features/inception/ZEN38/ACCAACGCTTATTTAT-1.npy',
 'example_data/data/image_features/inception/ZEN38/GCGGTGAACTGCGCTC-1.npy',
 'example_data/data/image_features/inception/ZEN38/CAGAGCATGAGCTTGC-1.npy']

Each spot is encoded with its unique barcode per slide id and the slide id itslef.